# Public reference patch control

Offline verification only. No model, no generated patches, no agent access to reference answers. Derived from the official starter bootstrap and verification API.

In [ ]:
import glob
import importlib
import logging
import os
import shutil
import subprocess
import sys
from pathlib import Path

# Configure environment variables for offline vLLM serving and LiteLLM routing
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['TRANSFORMERS_NO_FLAX'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['VLLM_NO_USAGE_STATS'] = '1'
os.environ['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'
os.environ['VLLM_ENGINE_READY_TIMEOUT_S'] = '1200'
os.environ['VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS'] = '1'
os.environ['NCCL_DEBUG'] = 'WARN'
os.environ['LITELLM_LOCAL_MODEL_COST_MAP'] = 'True'
os.environ['LITELLM_TELEMETRY'] = 'False'
os.environ['LITELLM_LOG'] = 'WARNING'
os.environ['OPENAI_API_KEY'] = os.environ.get('OPENAI_API_KEY', 'EMPTY')
os.environ['LOCAL_API_KEY'] = os.environ.get('LOCAL_API_KEY', 'EMPTY')
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['OTEL_SDK_DISABLED'] = 'true'
os.environ['OTEL_PYTHON_DISABLED'] = 'true'


def configure_logging() -> None:
    """Silence verbose third-party loggers."""
    for name in (
        'LiteLLM',
        'LiteLLM Router',
        'LiteLLM Proxy',
        'litellm',
        'httpx',
        'httpcore',
        'openai',
        'urllib3',
        'opentelemetry',
        'opentelemetry.context',
    ):
        lg = logging.getLogger(name)
        lg.setLevel(logging.WARNING)
        if 'litellm' in name.lower():
            lg.propagate = False
            lg.handlers.clear()


configure_logging()

WHEELHOUSE_DIR = Path('/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse')

# Remove broken cutlass .pth hooks if present
for pth_pattern in (
    '/usr/local/lib/python*/dist-packages/*cutlass*.pth',
    '/usr/local/lib/python*/site-packages/*cutlass*.pth',
):
    for pth in glob.glob(pth_pattern):
        try:
            os.unlink(pth)
        except OSError:
            pass

# Restore PEP 440 '+cu128' wheel filenames stripped by Kaggle dataset uploads
tmp_whl = Path('/tmp/wheelhouse')
tmp_whl.mkdir(parents=True, exist_ok=True)
for w in WHEELHOUSE_DIR.glob('*.whl'):
    if 'cutlass' in w.name.lower():
        continue
    target_name = (
        w.name.replace('cu128', '+cu128')
        if ('cu128' in w.name and '+' not in w.name)
        else w.name
    )
    target = tmp_whl / target_name
    if not target.exists():
        try:
            os.symlink(w, target)
        except OSError:
            shutil.copyfile(w, target)

wheels = sorted(str(w) for w in tmp_whl.glob('*.whl'))
print(f'Installing {len(wheels)} wheels from {WHEELHOUSE_DIR}...')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--force-reinstall', *wheels],
    check=True,
)
importlib.invalidate_caches()
configure_logging()
print('Wheelhouse installation complete.')

In [ ]:
import asyncio
import concurrent.futures
import json
import time
from adk_submission import ModelRegistry
from swegemma.config import EvalConfig
from swegemma.evaluate import Evaluator
from swegemma.models import load_tasks
from swegemma.deduplication import resolve_task_snapshot_paths
from swegemma.harness.verification import verify_task
from swegemma.results import append_task_result

DATA_DIR = Path('/kaggle/input/competitions/gemma-4-developer-agent')
OUT = Path('/kaggle/working/gold_control')
OUT.mkdir(parents=True, exist_ok=True)
config = EvalConfig(tasks_path=DATA_DIR/'tasks.jsonl', snapshots_dir=DATA_DIR/'snapshots',
    results_dir=OUT, submission_dir=OUT, models=ModelRegistry(), sandbox='subprocess',
    wheels_dir=DATA_DIR/'wheels', verification_timeout_seconds=300)
evaluator = Evaluator(config)
ids = {'fastapi_11355', 'fastapi_14512', 'fastapi_14953', 'fastapi_15661',
       'rich_3894', 'rich_3535', 'rich_3772', 'rich_3278',
       'requests_7505', 'requests_7427', 'requests_6757', 'httpx_3672'}
tasks = [t for t in load_tasks(config.tasks_path) if t.instance_id in ids]
assert len(tasks) == 12
results = []
for i, raw in enumerate(tasks, 1):
    task = evaluator._hydrate_task_from_secret(raw)
    assert task.patch.strip(), f'No public reference patch for {task.instance_id}'
    snapshot, base_snapshot, patch_path = resolve_task_snapshot_paths(config.snapshots_dir, task.instance_id, task.repo)
    print(f'[{i}/12] Public reference control: {task.instance_id}', flush=True)
    async def check():
        return await verify_task(evaluator.docker, config, task, snapshot,
            base_snapshot_path=base_snapshot, patch_path=patch_path,
            agent_patch=task.patch, start_time=time.perf_counter())
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
        result = pool.submit(lambda: asyncio.run(check())).result()
    results.append(result)
    append_task_result(result, OUT, results)
    print(f'  -> reference_resolved={result.resolved}, exit_code={result.test_exit_code}, error={result.error}', flush=True)
print('This control verifies published reference patches only. It does not measure agent quality.')
shutil.make_archive('/kaggle/working/gold_control_results', 'zip', OUT)
